In [1]:
import os

In [2]:
%pwd

'f:\\ML Sushmoy\\Detecting-Thyroid-Cancer Efficientnetb0\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'f:\\ML Sushmoy\\Detecting-Thyroid-Cancer Efficientnetb0'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class PrepareBaseModelConfig:
    root_dir: Path
    base_model_path: Path
    updated_base_model_path: Path
    params_image_size: list
    params_learning_rate: float
    params_include_top: bool
    params_weights: str
    params_classes: int

In [6]:
from ThyroidCancer.constants import *
from ThyroidCancer.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    
    def get_prepare_base_model_config(self) -> PrepareBaseModelConfig:
        config = self.config.prepare_base_model
        
        create_directories([config.root_dir])

        prepare_base_model_config = PrepareBaseModelConfig(
            root_dir=Path(config.root_dir),
            base_model_path=Path(config.base_model_path),
            updated_base_model_path=Path(config.updated_base_model_path),
            params_image_size=self.params.IMAGE_SIZE,
            params_learning_rate=self.params.LEARNING_RATE,
            params_include_top=self.params.INCLUDE_TOP,
            params_weights=self.params.WEIGHTS,
            params_classes=self.params.CLASSES
        )

        return prepare_base_model_config

In [8]:
import os
import tensorflow as tf

In [9]:
import tensorflow as tf
from pathlib import Path

class PrepareBaseModel:
    def __init__(self, config: PrepareBaseModelConfig):
        self.config = config

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        save_path = path.with_suffix(".h5")
        
        # THIS IS THE ONLY METHOD THAT WORKS 100% OF THE TIME
        model.save_weights(save_path)   # <--- ONLY SAVE WEIGHTS
        
        print(f"Model WEIGHTS successfully saved at: {save_path}")
        print("   → Architecture is fixed, so weights-only saving is safe & recommended")
    
    @staticmethod
    def load_weights(model: tf.keras.Model, weights_path: Path):
        model.load_weights(weights_path)
        print(f"Weights loaded successfully from: {weights_path}")

    def get_base_model(self):
        self.model = tf.keras.applications.EfficientNetB0(
            input_shape=self.config.params_image_size,
            weights=self.config.params_weights,
            include_top=self.config.params_include_top
        )
        print(f"Model loaded: {self.model.name}")

    @staticmethod
    def _prepare_full_model(model, classes, freeze_all, freeze_till, learning_rate):
        if freeze_all:
            model.trainable = False
        elif freeze_till is not None and freeze_till > 0:
            for layer in model.layers[:-freeze_till]:
                layer.trainable = False
            for layer in model.layers[-freeze_till:]:
                layer.trainable = True

        x = tf.keras.layers.GlobalAveragePooling2D(name="avg_pool")(model.output)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Dropout(0.5, name="top_dropout")(x)
        predictions = tf.keras.layers.Dense(
            units=classes,
            activation="softmax",
            name="predictions"
        )(x)

        full_model = tf.keras.models.Model(
            inputs=model.input,
            outputs=predictions,
            name="EfficientNetB0_transfer"
        )

        full_model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
            loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
            metrics=["accuracy"]
        )

        full_model.summary()
        return full_model

    def update_base_model(self):
        self.full_model = self._prepare_full_model(
            model=self.model,
            classes=self.config.params_classes,
            freeze_all=True,
            freeze_till=None,
            learning_rate=self.config.params_learning_rate
        )

        self.save_model(
            path=self.config.updated_base_model_path,
            model=self.full_model
        )

In [10]:
def fine_tune_model(self, unfreeze_from_layer=-30, learning_rate=1e-5):
    """
    Unfreeze the top N layers of EfficientNetB0 for fine-tuning.
    unfreeze_from_layer = -30 means last 30 layers are trainable (good starting point)
    """
    # Load the model that has the trained head
    model = tf.keras.models.load_model(self.config.updated_base_model_path)
    
    # The base EfficientNetB0 is wrapped inside the full model
    base_model = model.layers[0]  # EfficientNetB0 part (the backbone)
    # OR if you used name="EfficientNetB0_transfer" in Model(), you can do:
    # base_model = model.get_layer("nasnet_mobile")  # lowercase name

    base_model.trainable = True

    # Freeze all layers except the last N
    for layer in base_model.layers[:unfreeze_from_layer]:
        layer.trainable = False
    for layer in base_model.layers[unfreeze_from_layer:]:
        layer.trainable = True

    print(f"Fine-tuning: Unfreezing last {len(base_model.layers) + unfreeze_from_layer} layers")
    print("Trainable layers after unfreeze:")
    for layer in base_model.layers:
        if layer.trainable:
            print("  ✓", layer.name)

    # VERY IMPORTANT: Recompile with much lower learning rate
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),  # 1e-5 or 5e-5
        loss=tf.keras.losses.CategoricalCrossentropy(),
        metrics=["accuracy"]
    )

    model.summary()
    return model

In [11]:
try:
    config = ConfigurationManager()
    prepare_base_model_config = config.get_prepare_base_model_config()
    prepare_base_model = PrepareBaseModel(config=prepare_base_model_config)
    prepare_base_model.get_base_model()
    prepare_base_model.update_base_model()
except Exception as e:
    raise e

[2026-01-22 13:21:51,838: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-01-22 13:21:51,841: INFO: common: yaml file: params.yaml loaded successfully]
[2026-01-22 13:21:51,842: INFO: common: created directory at: artifacts]
[2026-01-22 13:21:51,852: INFO: common: created directory at: artifacts/prepare_base_model]
Model loaded: efficientnetb0
Model: "EfficientNetB0_transfer"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 rescaling (Rescaling)          (None, 224, 224, 3)  0           ['input_1[0][0]']                
    